# Section-A – Data Generation
## Introduction

In real-world data engineering projects, raw data is rarely perfect. To simulate practical business scenarios, this notebook generates synthetic e-commerce datasets containing intentional data quality issues such as missing values, invalid formats, negative quantities, and inconsistent text formatting.

The generated datasets will be used in the subsequent data cleaning, validation, and SQL analysis phases of the project.

## Objectives
Generate realistic e-commerce datasets using Python.
Create four CSV files:
orders.csv
order_items.csv
products.csv
customers.csv
Introduce controlled data quality issues.
Save the generated files in the data/raw/ directory.

## Libraries Required

In [ ]:
%pip install faker

In [2]:
import pandas as pd 
import random 
import os 
from faker import Faker 
from datetime import datetime, timedelta

## Initialize Faker and Random Seed

In [3]:
fake = Faker() 
random.seed(42)

## Create Raw Data Directory

In [5]:
os.makedirs('data/raw-data', exist_ok=True)

## Generate Customers Dataset
### Requirements
- At least 500 rows.
- Customer types: REGULAR, PREMIUM, VIP.
- About 2% invalid email addresses.

In [7]:
customers = [] 
for i in range(1, 501):
    email = fake.email()
# Introduce invalid emails (~2%) 
    if random.random() < 0.02: 
        email = email.replace('@', '') 
    
    customers.append([ 
        i, 
        fake.name(), 
        email, 
        fake.date_between(start_date='-2y', end_date='today'), 
        random.choice(['REGULAR', 'PREMIUM', 'VIP']) ]) 
    
customers_df = pd.DataFrame(customers, columns=[ 
    'customer_id', 
    'customer_name', 
    'email', 'registration_date', 
    'customer_type' ]) 

customers_df.to_csv('data/raw-data/customers.csv', index=False) 

print('customers.csv created') 
customers_df.head()

customers.csv created


,customer_id,customer_name,email,registration_date,customer_type
0,1,Marisa Wright,jlogan@example.net,2025-07-10,PREMIUM
1,2,Sonya Mccormick,erodriguez@example.net,2025-02-03,VIP
2,3,Kellie Knox,lgrimes@example.net,2024-10-28,REGULAR
3,4,Shane Aguilar,riverajonathan@example.com,2024-10-23,VIP
4,5,Edwin Delgado,nathan92@example.com,2025-08-15,PREMIUM


## Generate Products Dataset
### Requirements
- At least 500 rows.
- Categories such as Electronics, Clothing, Home, and Books.
- Some product names should contain extra spaces and mixed case.

In [4]:
categories = {
    'Electronics': ['Phone', 'Laptop', 'TV', 'Headphones'],
    'Clothing': ['Shirt', 'Jeans', 'Jacket', 'T-Shirt'],
    'Home': ['Table', 'Chair', 'Sofa', 'Lamp'],
    'Books': ['Novel', 'Biography', 'Comics', 'Dictionary']
} 

products = []

for i in range(1, 501):
    category = random.choice(list(categories.keys()))
    subcategory = random.choice(categories[category])
    
    product_name = f'{fake.word()} {subcategory}'
    
    # Introduce messy product names
    if random.random() < 0.10:
        product_name = ' ' + product_name.upper() + ' '
        
    products.append([
        f'P{i:04d}',
        product_name,
        category,
        subcategory,
        round(random.uniform(50, 5000), 2)
    ])
    
products_df = pd.DataFrame(products, columns=[
    'product_id',
    'product_name',
    'category',
    'subcategory',
    'cost_price' 
]) 
products_df.to_csv('data/raw-data/products.csv', index=False) 
print('products.csv created')
products_df.head()

products.csv created


,product_id,product_name,category,subcategory,cost_price
0,P0001,forward Phone,Electronics,Phone,1262.21
1,P0002,sort Shirt,Clothing,Shirt,4466.29
2,P0003,PROBABLY HEADPHONES,Electronics,Headphones,513.79
3,P0004,some Shirt,Clothing,Shirt,3594.30
4,P0005,college Biography,Books,Biography,1427.04


## Generate Orders Dataset
### Requirements
- At least 500 rows.
- Status values: PLACED, SHIPPED, DELIVERED, CANCELLED,RETURNED.
- About 5% missing customer_id.
- Some dates in incorrect DD-MM-YYYY format.

In [8]:
statuses = ['PLACED', 'SHIPPED', 'DELIVERED', 'CANCELLED', 'RETURNED'] 
regions = ['NORTH', 'SOUTH', 'EAST', 'WEST'] 
orders = [] 
start_date = datetime.now() - timedelta(days=365) 
for i in range(1, 501): 
    customer_id = random.randint(1, 500) 
    
    # Introduce missing customer_id (~5%) 
    if random.random() < 0.05: 
        customer_id = '' 
        
    order_date = start_date + timedelta(days=random.randint(0, 365)) 
    
    # Introduce wrong date format 
    if random.random() < 0.05: 
        order_date_str = order_date.strftime('%d-%m-%Y %H:%M:%S') 
        
    else: 
        order_date_str = order_date.strftime('%Y-%m-%d %H:%M:%S') 
        
    orders.append([
        i,
        customer_id,
        order_date_str,
        random.choice(statuses),
        random.choice(regions) ]) 
    
orders_df = pd.DataFrame(orders, columns=[
    'order_id',
    'customer_id',
    'order_date',
    'status',
    'region_code'
    ])

orders_df.to_csv('data/raw-data/orders.csv', index=False) 
print('orders.csv created') 
orders_df.head()

orders.csv created


,order_id,customer_id,order_date,status,region_code
0,1,345,08-05-2026 15:14:57,PLACED,WEST
1,2,69,2026-02-10 15:14:57,DELIVERED,NORTH
2,3,206,2026-05-21 15:14:57,RETURNED,EAST
3,4,38,2026-03-25 15:14:57,DELIVERED,NORTH
4,5,66,2026-02-24 15:14:57,DELIVERED,EAST


## Generate Order Items Dataset
### Requirements
- At least 500 rows.
- About 3% negative quantities.
- Ensure order_id exists in the orders table.

In [9]:
order_items = [] 

for i in range(1, 1501):
    quantity = random.randint(1, 5) 
    
    # Introduce negative quantities (~3%)
    if random.random() < 0.03:
        quantity = -quantity 
        
    order_items.append([
        i,
        random.randint(1, 500), # Existing order_id 
        f'P{random.randint(1, 500):04d}',
        quantity,
        round(random.uniform(100, 5000), 2), 
        round(random.uniform(0, 50), 2) ]) 
    
order_items_df = pd.DataFrame(order_items, columns=[
    'item_id',
    'order_id',
    'product_id',
    'quantity',
    'unit_price',
    'discount_percent'
    ])

order_items_df.to_csv('data/raw-data/order_items.csv', index=False)

print('order_items.csv created') 
order_items_df.head()

order_items.csv created


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,441,P0352,5,3901.93,22.86
1,2,236,P0162,1,3477.13,0.59
2,3,52,P0167,2,1871.85,43.09
3,4,122,P0138,2,2409.59,30.65
4,5,218,P0108,3,2153.56,37.61


## Verify Generated Files

In [5]:
import os 
files = os.listdir('data/raw-data')
print(files)

['customers.csv', 'orders.csv', 'order_items.csv', 'products.csv']


## Verify Row Counts

In [6]:
for file in files:
    path = f'data/raw-data/{file}'
    df = pd.read_csv(path) 
    print(file, len(df))

customers.csv 500
orders.csv 500
order_items.csv 1500
products.csv 500


## Conclusion

In this notebook, synthetic e-commerce datasets were successfully generated using Python and the Faker library. The generated datasets include realistic business data along with intentional data quality issues such as missing values, invalid formats, inconsistent text formatting, and negative quantities. These raw datasets have been saved in the `data/raw-data/` directory and will be used in the next notebook for data cleaning, validation, and analytical processing.